In [1]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# Example 1 - Covid Data

## 데이터 로드 및 전처리

## `df_covid19_100`

In [3]:
df_covid19 = pd.read_csv("owid-covid-data.csv", parse_dates=['date'])
df_covid19['date'] = pd.to_datetime(df_covid19['date'], format = "%Y-%m-%d") # 시간 객체
df_covid19.head()

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN


In [4]:
### 6개 대륙 값 선택
df_covid19_100 = df_covid19[(df_covid19['iso_code'].isin(['KOR',
                                                          'OWID_ASI',
                                                          'OWID_EUR',
                                                          'OWID_OCE',
                                                          'OWID_NAM',
                                                          'OWID_SAM',
                                                          'OWID_AFR']))]
### 최근 100일 선택
from datetime import timedelta

df_covid19_100 = df_covid19_100[(df_covid19['date'] >= (max(df_covid19['date']) - timedelta(days = 100)))]

In [5]:
### 지역명 (location) 한글로 변경

df_covid19_100.loc[df_covid19_100['location'] == 'South Korea', "location"] = '한국'
df_covid19_100.loc[df_covid19_100['location'] == 'Asia', "location"] = '아시아'
df_covid19_100.loc[df_covid19_100['location'] == 'Europe', "location"] = '유럽'
df_covid19_100.loc[df_covid19_100['location'] == 'Oceania', "location"] = '오세아니아'
df_covid19_100.loc[df_covid19_100['location'] == 'North America', "location"] = '북미'
df_covid19_100.loc[df_covid19_100['location'] == 'South America', "location"] = '남미'
df_covid19_100.loc[df_covid19_100['location'] == 'Africa', "location"] = '아프리카'


### 순서 범주형 데이터로 변경
from pandas.api.types import CategoricalDtype

ord = CategoricalDtype(categories = ['한국', '아시아', '유럽', '북미', '남미', '아프리카','오세아니아'], ordered = True)
print(ord)
df_covid19_100['location'] = df_covid19_100['location'].astype(ord)

category


In [6]:
### 날짜순으로 정렬
df_covid19_100 = df_covid19_100.sort_values(by = 'date')
df_covid19_100.head()

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
3257,OWID_AFR,NaN,아프리카,2024-05-06,13142692.0,0.0,27.43,259103.0,0.0,0.00,...,NaN,NaN,NaN,NaN,NaN,1426736614,NaN,NaN,NaN,NaN
289201,OWID_OCE,NaN,오세아니아,2024-05-06,14930974.0,0.0,987.57,32576.0,0.0,3.86,...,NaN,NaN,NaN,NaN,NaN,45038860,NaN,NaN,NaN,NaN
120152,OWID_EUR,NaN,유럽,2024-05-06,252559092.0,0.0,2330.57,2100573.0,0.0,11.29,...,NaN,NaN,NaN,NaN,NaN,744807803,NaN,NaN,NaN,NaN
359147,OWID_SAM,NaN,남미,2024-05-06,68798132.0,0.0,80.71,1354126.0,0.0,2.86,...,NaN,NaN,NaN,NaN,NaN,436816679,NaN,NaN,NaN,NaN
21675,OWID_ASI,NaN,아시아,2024-05-06,301417754.0,0.0,847.43,1636990.0,0.0,5.29,...,NaN,NaN,NaN,NaN,NaN,4721383370,NaN,NaN,NaN,NaN


## `df_covid19_stat`

In [8]:
### 국가별 Covid 통계량
df_covid19_stat = df_covid19.groupby(['iso_code', 'continent', 'location'], dropna = True)


df_covid19_stat = df_covid19_stat.agg(
        인구수 = ('population', 'max'),
        전체사망자수 = ('new_deaths', 'sum'),
        백신접종자완료자수 = ('people_fully_vaccinated', 'max'),
        인구백명당백신접종완료율 = ('people_fully_vaccinated_per_hundred', 'max'),
        인구백명당부스터접종자수 = ('total_boosters_per_hundred', 'max'))

df_covid19_stat = df_covid19_stat.reset_index()
df_covid19_stat.head()


,iso_code,continent,location,인구수,전체사망자수,백신접종자완료자수,인구백명당백신접종완료율,인구백명당부스터접종자수
0,ABW,North America,Aruba,106459,292.0,84368.0,79.25,NaN
1,AFG,Asia,Afghanistan,41128772,7998.0,18370386.0,44.67,6.64
2,AGO,Africa,Angola,35588996,1937.0,9609080.0,27.00,8.62
3,AIA,North America,Anguilla,15877,12.0,10380.0,65.38,20.84
4,ALB,Europe,Albania,2842318,3605.0,1279333.0,45.01,14.16


In [9]:
### 10만명당 사망자 수
df_covid19_stat['십만명당사망자수'] = round(df_covid19_stat['전체사망자수'] / df_covid19_stat['인구수'] * 100000, 1)

### 백신접종 완료율
df_covid19_stat['백신접종완료율'] = df_covid19_stat['백신접종자완료자수'] / df_covid19_stat['인구수']


### Q1: 인구수가 100만명 이상인 국가 중 백신 접종률이 가장 높은 나라는?

In [10]:
# 인구수가 백만명 이상의 국가
vaccine_top10 = df_covid19_stat.loc[df_covid19_stat['인구수'] > 10000000]
vaccine_top10 = vaccine_top10.sort_values(by = ['인구백명당백신접종완료율'], ascending = False).head(10)
vaccine_top10 = vaccine_top10.reset_index()
vaccine_top10

,index,iso_code,continent,location,인구수,전체사망자수,백신접종자완료자수,인구백명당백신접종완료율,인구백명당부스터접종자수,십만명당사망자수,백신접종완료율
0,37,CHL,South America,Chile,19603736,66162.0,1.770012e+07,90.29,140.15,337.5,0.902895
1,48,CUB,North America,Cuba,11212198,8530.0,1.005366e+07,89.67,78.42,76.1,0.896671
2,38,CHN,Asia,China,1425887360,122326.0,1.276760e+09,89.54,57.99,8.6,0.895414
3,111,KHM,Asia,Cambodia,16767851,3056.0,1.469327e+07,87.63,64.31,18.2,0.876277
4,235,VNM,Asia,Vietnam,98186856,43206.0,8.596156e+07,87.55,59.05,44.0,0.875490
5,223,TWN,Asia,Taiwan,23893396,0.0,2.079309e+07,87.02,106.58,0.0,0.870244
6,179,PRT,Europe,Portugal,10270857,28809.0,8.909769e+06,86.75,68.99,280.5,0.867481
7,63,ESP,Europe,Spain,47558632,121852.0,4.073912e+07,85.66,55.85,256.2,0.856608
8,114,KOR,Asia,South Korea,51815808,35934.0,4.437268e+07,85.64,79.76,69.3,0.856354
9,172,PER,South America,Peru,34049588,220975.0,2.870964e+07,84.32,94.39,649.0,0.843172


In [11]:
import plotly.graph_objects as go
fig = go.Figure()

for continent, group in vaccine_top10.groupby('continent'): # (그룹명:continent, 데이터:group)
    fig.add_trace(go.Bar(
        x = group['location'],
        y = group['인구백명당백신접종완료율'],
        name = continent,
        text = group['인구백명당백신접종완료율'],
        textposition = 'outside',
        texttemplate = '%{text}%')
                 )
fig.update_layout(
    title = dict(text = '완전 백신 접종률 상위 top 10 국가', x = 0.5),
    xaxis = dict(title = '국가', categoryorder = 'total descending'), # 정렬
    yaxis = dict(title = '백신접종완료율', ticksuffix = '%', range = (0, 100)))

fig.show()

### Q2: 대륙별로 접종완료율이 가장 높은 나라 5개국은?

In [12]:
### 대륙별 상위 5개국 (인구 100만 이상)
temp = df_covid19_stat.loc[df_covid19_stat['인구수'] > 10000000]
temp = temp.sort_values(by = ['continent', '인구백명당백신접종완료율'],
                        ascending = False).groupby('continent')
vaccine_top5_by_continent = temp.head(5).reset_index()
vaccine_top5_by_continent


,index,iso_code,continent,location,인구수,전체사망자수,백신접종자완료자수,인구백명당백신접종완료율,인구백명당부스터접종자수,십만명당사망자수,백신접종완료율
0,37,CHL,South America,Chile,19603736,66162.0,1.770012e+07,90.29,140.15,337.5,0.902895
1,172,PER,South America,Peru,34049588,220975.0,2.870964e+07,84.32,94.39,649.0,0.843172
2,29,BRA,South America,Brazil,215313504,702116.0,1.761642e+08,81.82,58.70,326.1,0.818175
3,59,ECU,South America,Ecuador,18001002,36050.0,1.424059e+07,79.11,58.64,200.3,0.791100
4,7,ARG,South America,Argentina,45510324,130663.0,3.490061e+07,76.69,82.87,287.1,0.766872
5,11,AUS,Oceania,Australia,26177410,25312.0,2.164752e+07,82.70,75.60,96.7,0.826954
6,175,PNG,Oceania,Papua New Guinea,10142625,700.0,3.211920e+05,3.17,0.34,6.9,0.031668
7,48,CUB,North America,Cuba,11212198,8530.0,1.005366e+07,89.67,78.42,76.1,0.896671
8,35,CAN,North America,Canada,38454328,55613.0,3.175825e+07,82.59,90.73,144.6,0.825869
9,228,USA,North America,United States,338289856,1193165.0,2.306373e+08,69.47,40.08,352.7,0.681774


In [13]:
fig = go.Figure()

for continent, group in vaccine_top5_by_continent.groupby('continent'):
    fig.add_trace(go.Bar(
        y = group['location'],
        x = group['인구백명당백신접종완료율'],
        name = continent,
        text = group['인구백명당백신접종완료율'],
        textposition = 'outside',
        texttemplate = '%{text}%',
        orientation = 'h'))

fig.update_layout(title = dict(text = '대륙별 완전 백신 접종률 상위 top 5 국가', x = 0.5),
                  xaxis = dict(title = '백신접종완료율',
                               ticksuffix = '%',
                               range = (0, 105)),
                  yaxis = dict(title = '',
                               autorange = 'reversed'),
                  margin = {'t' : 50, 'b' : 25, 'l' : 25, 'r' : 25},
                  height = 800)
fig.show()

In [14]:
# 100만명당 사망자 수
fig.add_trace(
    go.Scatter(mode = 'markers+text',
               xaxis = "x2", # 두번째 축
               name = '사망자수',
               x = vaccine_top5_by_continent['십만명당사망자수'],
               y = vaccine_top5_by_continent['location'],
               text = round(vaccine_top5_by_continent['십만명당사망자수'], 1),
               textposition = 'middle right'))

fig.update_layout(title = dict(text = '대륙별 완전 백신 접종률 상위 top 5 국가', x = 0.5),
                  xaxis = dict(title = '백신접종완료율',
                               ticksuffix = '%',
                               range = (0, 105)),
                  xaxis2 = dict(title = dict(text = '인구10만명당 사망자수', # xaxis2 축의 설정
                                            standoff = 0),
                                side = "top",
                                overlaying = "x",
                                range = (0, 700),
                                ticksuffix = '명'),
                  yaxis = dict(title = '',
                               autorange = 'reversed'),
                 margin = {'t' : 100, 'r' : 100}, height = 800
                 )

###  Q3. 대륙별 백신 접종율에 차이가 있는가?

In [15]:
df_covid19_stat2 = df_covid19.groupby(['iso_code', 'continent', 'location'], dropna = False)


df_covid19_stat2 = df_covid19_stat2.agg(
        인구수 = ('population', 'max'),
        전체사망자수 = ('new_deaths', 'sum'),
        백신접종자완료자수 = ('people_fully_vaccinated', 'max'),
        인구백명당백신접종완료율 = ('people_fully_vaccinated_per_hundred', 'max'),
        인구백명당부스터접종자수 = ('total_boosters_per_hundred', 'max'))

df_covid19_stat2 = df_covid19_stat2.reset_index()
df_covid19_stat2

,iso_code,continent,location,인구수,전체사망자수,백신접종자완료자수,인구백명당백신접종완료율,인구백명당부스터접종자수
0,ABW,North America,Aruba,106459,292.0,84368.0,79.25,NaN
1,AFG,Asia,Afghanistan,41128772,7998.0,18370386.0,44.67,6.64
2,AGO,Africa,Angola,35588996,1937.0,9609080.0,27.00,8.62
3,AIA,North America,Anguilla,15877,12.0,10380.0,65.38,20.84
4,ALB,Europe,Albania,2842318,3605.0,1279333.0,45.01,14.16
...,...,...,...,...,...,...,...,...
250,WSM,Oceania,Samoa,222390,31.0,177954.0,80.02,36.99
251,YEM,Asia,Yemen,33696612,2159.0,807057.0,2.40,0.20
252,ZAF,Africa,South Africa,59893884,102595.0,21038797.0,35.13,7.36
253,ZMB,Africa,Zambia,20017670,4077.0,9213802.0,46.03,0.17


In [16]:
df_radar_veccine = df_covid19_stat2[(df_covid19_stat2['iso_code'].isin(['OWID_ASI',
                                                                        'OWID_EUR',
                                                                        'OWID_OCE',
                                                                        'OWID_NAM',
                                                                        'OWID_SAM',
                                                                        'OWID_AFR']))]
df_radar_veccine

,iso_code,continent,location,인구수,전체사망자수,백신접종자완료자수,인구백명당백신접종완료율,인구백명당부스터접종자수
163,OWID_AFR,NaN,Africa,1426736614,259121.0,4.624114e+08,32.41,6.86
164,OWID_ASI,NaN,Asia,4721383370,1637335.0,3.462095e+09,73.33,38.45
168,OWID_EUR,NaN,Europe,744807803,2102377.0,4.937513e+08,66.29,49.02
173,OWID_NAM,NaN,North America,600323657,1671512.0,3.944939e+08,65.71,42.69
175,OWID_OCE,NaN,Oceania,45038860,33024.0,2.807290e+07,62.33,56.40
176,OWID_SAM,NaN,South America,436816679,1357619.0,3.371179e+08,77.18,58.95


In [17]:
# Scatterpolar Radar chart
fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    theta = df_radar_veccine['location'], # 각 (angle)
    r = df_radar_veccine['인구백명당백신접종완료율'], # 반지름 (radius)
    fill = 'toself'
    ))

fig.update_layout(polar =
    dict(
        angularaxis =  dict(
           ticktext = ('아프리카', '아시아', '유럽', '북미', '오세아니아', '남미'),
           tickvals = ('Africa', 'Asia', 'Europe', 'North America', 'Oceania', 'South America'),
           linewidth = 2,
           linecolor = 'black',
           gridcolor = 'gray'),
       radialaxis = dict(
           linewidth = 2,
           linecolor = 'dodgerblue',
           gridcolor = 'skyblue',
           nticks = 5,
           ticksuffix = '%',
           title = '백신 접종률')
    ),
    title = dict(text = '대륙별 백신 접종률', x = 0.5))

fig.show()

### Q4: 소득 구간별 코로나 대응 상황(백신접종자수 / 사망자수) 비교- 비율 그래프 (Bar Chart의 변형)

In [25]:
df_covid19_stat2 # exsiting data

df_covid19_stat3 = df_covid19_stat2.loc[df_covid19_stat2['iso_code'].isin(
    ['OWID_HIC', 'OWID_LIC', 'OWID_LMC', 'OWID_UMC']),
    ['location', '전체사망자수', '백신접종자완료자수',
     '인구백명당백신접종완료율']].transpose()

df_covid19_stat3

,169,171,172,178
location,High-income countries,Low-income countries,Lower-middle-income countries,Upper-middle-income countries
전체사망자수,3001093.0,43530.0,1188056.0,2824538.0
백신접종자완료자수,929255961.0,204944352.0,2053053994.0,1990653301.0
인구백명당백신접종완료율,74.31,27.79,59.82,78.81


In [26]:
df_covid19_stat3 = df_covid19_stat3.rename(columns = df_covid19_stat3.iloc[0]).drop(df_covid19_stat3.index[0])
df_covid19_stat3

,High-income countries,Low-income countries,Lower-middle-income countries,Upper-middle-income countries
전체사망자수,3001093.0,43530.0,1188056.0,2824538.0
백신접종자완료자수,929255961.0,204944352.0,2053053994.0,1990653301.0
인구백명당백신접종완료율,74.31,27.79,59.82,78.81


In [27]:
df_covid19_stat3['sum'] = df_covid19_stat3.sum(axis = 1) # 전체

### 비율계산
df_covid19_stat3['High income'] = df_covid19_stat3['High-income countries'] / df_covid19_stat3['sum']
df_covid19_stat3['Low income'] = df_covid19_stat3['Low-income countries'] / df_covid19_stat3['sum']
df_covid19_stat3['Lower middle income'] = df_covid19_stat3['Lower-middle-income countries'] / df_covid19_stat3['sum']
df_covid19_stat3['Upper middle income'] = df_covid19_stat3['Upper-middle-income countries'] / df_covid19_stat3['sum']
df_covid19_stat3.drop(columns = ["High-income countries",	"Low-income countries",	"Lower-middle-income countries",	"Upper-middle-income countries"], inplace = True)
df_covid19_stat3

,sum,High income,Low income,Lower middle income,Upper middle income
전체사망자수,7057217.0,0.425252,0.006168,0.168346,0.400234
백신접종자완료자수,5177907608.0,0.179466,0.039581,0.396503,0.384451
인구백명당백신접종완료율,240.73,0.308686,0.115441,0.248494,0.327379


In [28]:
fig = go.Figure()

# 'High income'
fig.add_trace(go.Bar(
    x = df_covid19_stat3['High income'],
    y = df_covid19_stat3.index,
    orientation = 'h',
    name = 'High income',
    marker = dict(line = dict(color = 'white', width = 2))
    ))

## 'Upper middle income'
fig.add_trace(go.Bar(
    x = df_covid19_stat3['Upper middle income'],
    y = df_covid19_stat3.index,
    orientation = 'h',
    name = 'Upper middle income',
    marker = dict(line = dict(color = 'white', width = 2))
    ))

## 'Lower middle income'
fig.add_trace(go.Bar(
    x = df_covid19_stat3['Lower middle income'],
    y = df_covid19_stat3.index,
    orientation = 'h',
    name = 'Lower middle income',
    marker = dict(line = dict(color = 'white', width = 2))
    ))

## 'Low income'
fig.add_trace(go.Bar(
    x = df_covid19_stat3['Low income'],
    y = df_covid19_stat3.index,
    orientation = 'h',
    name = 'Low income',
    marker = dict(line = dict(color = 'white', width = 2))
    ))
fig.show()


In [29]:
# Bar Chart 의 변형
fig.update_layout(barmode = 'stack', # 비율
    title = dict(text = '국가 소득 구간별 코로나19 현황', x = 0.5),
    xaxis = dict(title = '', tickformat = '.0%'),
    yaxis = dict(title = ''),
    legend = dict(orientation = 'h'),
    margin =  {'t' : 50, 'b' : 25, 'l' : 25, 'r' : 25})

for index, row in df_covid19_stat3.iterrows():
    # 'High income' 주석
    fig.add_annotation(
        xref = 'x',
        yref = 'y',
        x = row['High income'] / 2,
        y = index,
        text = str(round(row['High income'] * 100, 1)) + '%',
        showarrow = False,
        font = dict(color = 'white'))

    # 'Upper middle income' 주석
    fig.add_annotation(
        xref = 'x',
        yref = 'y',
        x = (row['Upper middle income'] / 2) + row['High income'],
        y = index,
        text = str(round(row['Upper middle income'] * 100, 1)) + '%',
        showarrow = False,
        font = dict(color = 'white'))

    # 'Lower middle income' 주석
    fig.add_annotation(
        xref = 'x',
        yref = 'y',
        x = (row['Lower middle income'] / 2) + row['High income'] +
        row['Upper middle income'], y = index,
        text = str(round(row['Lower middle income'] * 100, 1)) + '%',
        showarrow = False,
        font = dict(color = 'white'))

    # 'Low income' 주석
    fig.add_annotation(
        xref = 'x',
        yref = 'y',
        x = (row['Low income'] / 2) + row['High income'] + row['Upper middle income'] + row['Lower middle income'],
        y = index,
        text = str(round(row['Low income'] * 100, 1)) + '%',
        showarrow = False,
        font = dict(color = 'white'))

fig.show()

### Q5: 주요 5개국의 사망자수 추이 - Line Graph의 활용 (Hover기능)

In [30]:
df_covid19_5 = df_covid19.copy()

# 한국, 미국, 일본, 독일, 프랑스 데이터
df_covid19_5 = df_covid19_5[(df_covid19_5['iso_code'].isin(['KOR',
                                                            'USA',
                                                            'JPN',
                                                            'GBR',
                                                            'FRA']))]
df_covid19_5.dropna(subset = ['total_deaths_per_million'])
df_covid19_5

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
130367,FRA,Europe,France,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,35.6,NaN,5.98,82.66,0.90,67813000,-1002.2,-7.29,-7.29,-15.54
130368,FRA,Europe,France,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,35.6,NaN,5.98,82.66,0.90,67813000,NaN,NaN,NaN,NaN
130369,FRA,Europe,France,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,35.6,NaN,5.98,82.66,0.90,67813000,NaN,NaN,NaN,NaN
130370,FRA,Europe,France,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,35.6,NaN,5.98,82.66,0.90,67813000,NaN,NaN,NaN,NaN
130371,FRA,Europe,France,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,35.6,NaN,5.98,82.66,0.90,67813000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405120,USA,North America,United States,2024-07-31,103436829.0,NaN,NaN,1192546.0,0.0,88.43,...,24.6,NaN,2.77,78.86,0.93,338289856,NaN,NaN,NaN,NaN
405121,USA,North America,United States,2024-08-01,103436829.0,NaN,NaN,1192546.0,0.0,88.43,...,24.6,NaN,2.77,78.86,0.93,338289856,NaN,NaN,NaN,NaN
405122,USA,North America,United States,2024-08-02,103436829.0,NaN,NaN,1192546.0,0.0,88.43,...,24.6,NaN,2.77,78.86,0.93,338289856,NaN,NaN,NaN,NaN
405123,USA,North America,United States,2024-08-03,103436829.0,NaN,NaN,1192546.0,0.0,88.43,...,24.6,NaN,2.77,78.86,0.93,338289856,NaN,NaN,NaN,NaN


In [31]:
fig = go.Figure()

### Error Fixed part
nations = {
    'South Korea': 'solid',
    'United States': 'dot',
    'Japan': 'dash',
    'United Kingdom': 'dashdot',
    'France': 'longdash'
}

for location, group in df_covid19_5.groupby('location'):
    fig.add_trace(go.Scatter(
        mode = 'lines',
        x = group['date'],
        y = group['total_deaths_per_million'],
        line = dict(dash = nations[location]),
        name = location,
        connectgaps = True))

fig.update_layout(
    title = dict(text = '코로나19 사망자수 추세', x = 0.5),
    xaxis = dict(title = '', spikemode = 'across',
    ## rangeslider
    rangeslider = dict(visible = True),
    rangeselector = dict(
                     buttons = list([
                         dict(count = 7,
                              label = '1 Week before',
                              step = 'day',
                              stepmode = "backward"),
                         dict(count = 1,
                              label = '1 month before',
                              step = 'month',
                              stepmode = "backward"),
                         dict(count  = 6,
                              label = '6 months before',
                              step = "month",
                              stepmode = "backward"),
                         dict(count = 1,
                              label = '1 year before',
                              step = "year",
                              stepmode = "backward")
    ]))),
    yaxis = dict(title = '10만명당 사망자수 누계', spikemode = 'toaxis'),
    hovermode = "x unified"
)

fig.show()

# Example 2 - 취업 통계 데이터

## 데이터 로드 및 전처리

In [32]:
# 구글드라이브에서 데이터 불러오기
## 출처 : https://kess.kedi.re.kr/contents/dataset?itemCode=04&menuId=m_02_04_03_02&tabId=m3, 학과별, 2021년도
### NOTE : 해당 파일(주피터 노트북, ipynb 파일)을 구글코랩 환경이 아닌 다른 환경에서 실행하는 경우, os.chdir() 에 들어가는 작업디렉토리를 실습 데이터 (엑셀파일)가 위치한 디렉토리로 변경후 사용하시기 바랍니다.
### NOTE : 실습 데이터는 제공되며, 위 사이트에서 다운로드 받으실 수 있습니다.

#from google.colab import drive
#import os
#drive.mount('/content/drive')

#os.chdir("/content/drive/MyDrive/데이터가공과시각화[2]")
#file_path = "./2021년 학과별 고등교육기관 취업통계.xlsx"

df_취업률 = pd.read_excel(
    "2021년 학과별 고등교육기관 취업통계.xlsx",
    sheet_name="학과별",
    skiprows=13,  # 첫 13행을 제외
    dtype={col: "string" for col in range(9)}  # 처음 9개 컬럼을 문자형으로 지정
)
df_취업률.head()

,조사기준일,학제,과정구분,대계열,중계열,소계열,학과코드,학과명,학위구분,졸업자_계,...,3차 유지취업률_여,4차 유지취업자_계,4차 유지취업자_남,4차 유지취업자_여,4차 유지취업률_계,4차 유지취업률_남,4차 유지취업률_여,입학당시 기취업자_계,입학당시 기취업자_남,입학당시 기취업자_여
0,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100002,관광일본어과,<NA>,79,...,50.0,5,1,4,38.5,20.0,50.0,0,0,0
1,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100003,관광일본어전공,<NA>,26,...,100.0,7,4,3,87.5,80.0,100.0,1,1,0
2,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100005,관광일어과,<NA>,107,...,59.1,25,12,13,59.5,60.0,59.1,4,1,3
3,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100011,실무일본어과,<NA>,109,...,60.0,21,0,21,52.5,0.0,52.5,4,0,4
4,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100013,일본어과,<NA>,63,...,50.0,8,3,5,40.0,30.0,50.0,4,1,3


In [33]:
df_취업률 = pd.concat([df_취업률.iloc[:, 0:8], # 1~9열 선택
                     df_취업률.loc[:, df_취업률.columns.str.endswith('계')], # '계'로 끝나는 변수
                     df_취업률.loc[:, '입대자']],                           # '입대자' 변수
                     axis = 1) # 변수 (열방향 선택)
df_취업률

,조사기준일,학제,과정구분,대계열,중계열,소계열,학과코드,학과명,졸업자_계,취업률_계,...,1차 유지취업자_계,1차 유지취업률_계,2차 유지취업자_계,2차 유지취업률_계,3차 유지취업자_계,3차 유지취업률_계,4차 유지취업자_계,4차 유지취업률_계,입학당시 기취업자_계,입대자
0,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100002,관광일본어과,79,29.6,...,10,76.9,8,61.5,7,53.8,5,38.5,0,14
1,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100003,관광일본어전공,26,61.1,...,7,87.5,7,87.5,7,87.5,7,87.5,1,0
2,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100005,관광일어과,107,50.0,...,34,81.0,32,76.2,27,64.3,25,59.5,4,3
3,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100011,실무일본어과,109,52.3,...,29,72.5,27,67.5,24,60.0,21,52.5,4,0
4,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100013,일본어과,63,46.2,...,14,70.0,13,65.0,8,40.0,8,40.0,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9401,2021.12.31,기능대학,전문대학과정,예체능계열,디자인,기타디자인,C07010400310,주얼리디자인과,39,81.1,...,15,75.0,13,65.0,10,50.0,10,50.0,1,0
9402,2021.12.31,기능대학,전문대학과정,예체능계열,디자인,기타디자인,C07010400403,융합디자인과,33,87.1,...,15,68.2,15,68.2,15,68.2,15,68.2,0,1
9403,2021.12.31,기능대학,전문대학과정,예체능계열,디자인,기타디자인,C07010410015,스마트 제품디자인과,21,83.3,...,9,90.0,8,80.0,6,60.0,5,50.0,0,3
9404,2021.12.31,기능대학,전문대학과정,예체능계열,응용예술,영상·예술,C07020300014,영상그래픽과,14,84.6,...,7,100.0,7,100.0,6,85.7,6,85.7,0,1


In [41]:
## df_취업률에서 졸업자가 500명 이하인 학과 중 25% 샘플링
df_취업률_500 = df_취업률.loc[(df_취업률['졸업자_계'] < 500)]
df_취업률_500 = df_취업률_500.iloc[range(0, len(df_취업률_500.index) , 4)]
df_취업률_500 = df_취업률_500.rename(columns = {'졸업자_계':'졸업자수', '취업률_계': '취업률', '취업자_합계_계':'취업자수'})
df_취업률_500

,조사기준일,학제,과정구분,대계열,중계열,소계열,학과코드,학과명,졸업자수,취업률,...,1차 유지취업자_계,1차 유지취업률_계,2차 유지취업자_계,2차 유지취업률_계,3차 유지취업자_계,3차 유지취업률_계,4차 유지취업자_계,4차 유지취업률_계,입학당시 기취업자_계,입대자
0,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100002,관광일본어과,79,29.6,...,10,76.9,8,61.5,7,53.8,5,38.5,0,14
4,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100013,일본어과,63,46.2,...,14,70.0,13,65.0,8,40.0,8,40.0,4,4
8,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,일본어,C01010100049,항공호텔관광학부 일본어통역전공,40,37.5,...,5,62.5,5,62.5,5,62.5,5,62.5,0,2
12,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,중국어,C01010200003,관광중국어과,225,51.9,...,68,84.0,62,76.5,53,65.4,52,64.2,6,9
16,2021.12.31,전문대학,전문대학과정,인문계열,언어ㆍ문학,중국어,C01010200011,중국어과,5,60.0,...,3,100.0,2,66.7,2,66.7,2,66.7,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9388,2021.12.31,기능대학,전문대학과정,자연계열,생물ㆍ화학ㆍ환경,생물,C05020100066,바이오품질관리과,28,89.3,...,25,100.0,25,100.0,23,92.0,23,92.0,0,0
9392,2021.12.31,기능대학,전문대학과정,의약계열,치료ㆍ보건,의료장비,C06020300002,의료공학과,1,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
9396,2021.12.31,기능대학,전문대학과정,예체능계열,디자인,패션디자인,C07010300011,패션디자인과,21,87.5,...,8,80.0,6,60.0,5,50.0,4,40.0,0,2
9400,2021.12.31,기능대학,전문대학과정,예체능계열,디자인,기타디자인,C07010400214,금형디자인과,326,80.3,...,209,89.7,197,84.5,184,79.0,178,76.4,15,22


### Q1: 졸업자 대비 취업자 수 (Not really interesting, but for illustration)

In [35]:
# 컬러 설정용
colors = {'의약계열': 0,
          '인문계열': 1,
          '사회계열': 2,
          '교육계열': 3,
          '공학계열': 4,
          '자연계열': 5,
          '예체능계열': 6}

fig = go.Figure()

for cat, group in df_취업률_500.groupby('대계열'):
    fig.add_trace(go.Scatter(
        mode = 'markers',
        x = group['졸업자수'],
        y = group['취업자수'],
        name = cat,
        marker = dict(color = colors[cat]), # colors 딕셔너리 참조
        showlegend = True
   ))

    fig.update_layout(
    title = dict(text = '<b>졸업자 대비 취업자수</b>',
                 x = 0.5),
    margin = {'t' : 50, 'b' : 25, 'l' : 25, 'r' : 25},
    xaxis = dict(color = 'white',
                 ticksuffix = '명',
                 showgrid = False),
    yaxis = dict(color = 'white',
                 gridcolor = 'gray',
                 ticksuffix = '명',
                 dtick = 100))
fig.show()

In [37]:
# 추세선의 추가
## 선형회귀모형
from sklearn.linear_model import LinearRegression
linear_regr = LinearRegression()

X = df_취업률_500['졸업자수'].values.reshape(-1,1)
Y = df_취업률_500['취업자수'].values

linear_regr.fit(X, Y)
linear_fit = linear_regr.predict(X)
linear_fit

array([ 46.73186536,  37.17784169,  23.44393267, ...,  12.09852957,
       194.22210569,   7.91864422], shape=(2313,))

In [38]:
# LOWESS
import statsmodels.api as sm
lowess_fit = sm.nonparametric.lowess(df_취업률_500['취업자수'], df_취업률_500['졸업자수'])
lowess_fit

array([[  1.        ,   0.69513624],
       [  1.        ,   0.69513624],
       [  1.        ,   0.69513624],
       ...,
       [493.        , 298.48020199],
       [495.        , 299.70628459],
       [496.        , 300.31932406]], shape=(2313, 2))

In [39]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    mode = 'markers',
    x = df_취업률_500['졸업자수'],
    y = df_취업률_500['취업자수'],
    showlegend = False))

# Linear Regression
fig.add_trace(go.Scatter(
    mode = 'lines',
    x = df_취업률_500['졸업자수'],
    y = linear_fit,
    name = '선형회귀적합선',
    line = dict(dash = 'dot')))

# LOWESS
fig.add_trace(go.Scatter(
    mode = 'lines',
    x = lowess_fit[:,0],
    y = lowess_fit[:,1],
    name = 'LOWESS'))

fig.update_layout(
    title = dict(text = '<b>졸업자 대비 취업자수</b>',
                 x = 0.5),
    margin = {'t' : 50, 'b' : 25, 'l' : 25, 'r' : 25},
    xaxis = dict(color = 'white',
                 ticksuffix = '명',
                 showgrid = False),
    yaxis = dict(color = 'white',
                 gridcolor = 'gray',
                 ticksuffix = '명',
                 dtick = 100))

fig.show()

### Q2: (대) 계열별로 졸업자수는 어떻게 될까? - 분포의 확인 (파이차트)

In [43]:
df_pie = df_취업률_500.groupby('대계열')['졸업자수'].sum().reset_index()
df_pie

,대계열,졸업자수
0,공학계열,27894
1,교육계열,3701
2,사회계열,18450
3,예체능계열,15383
4,의약계열,3950
5,인문계열,6823
6,자연계열,12079


In [44]:
fig = go.Figure()

fig.add_trace(go.Pie(
    values = df_pie['졸업자수'],
    labels = df_pie['대계열'],
    direction = 'clockwise',
    sort = True,
    hole = 0.3, # 도넛
    pull = (0, 0.2, 0, 0, 0, 0, 0))) # 강조

fig.update_layout(title = dict(text = '대학 계열별 졸업생 분포', x = 0.5))

fig.add_annotation(x = 0.5,
                   y = 0.5,
                   text = '<b>졸업생수</b>',
                   showarrow = False,
                   xanchor = 'center',
                   font = dict(size = 20)
                  )
fig.show()

In [45]:
## 총 졸업자수
전체_sum = df_취업률_500['졸업자수'].sum()
## 대계열 별 졸업자 수
계열_sum = df_취업률_500.groupby('대계열').sum().reset_index()['졸업자수'].values.tolist()
계열_sum

## (대 > 중)계열별 졸업자수
중계열_sum = df_취업률_500.groupby(['대계열', '중계열']).sum()['졸업자수'].reset_index()

In [46]:
# 썬버스트 차트
name1 = ['전체']                             # 1st layer 이름
name2 = 중계열_sum['대계열'].unique().tolist() # 2nd layer 이름
name3 = 중계열_sum['중계열'].tolist()          # 3rd layer 이름

In [47]:
fig = go.Figure()

fig.add_trace(go.Sunburst(
    # 트레이스의 labels
    labels = name1 +
             name2 +
             name3,
    # 트레이스의 parents
    parents = [''] +
              ['전체'] * len(name2) +
              중계열_sum['대계열'].tolist(),
    # 트레이스의 values
    values = [전체_sum] +
             계열_sum +
             중계열_sum['졸업자수'].values.tolist(),
    branchvalues = 'total', # 'total' or 'remainder'
    insidetextorientation = 'radial')) # 'radial' or 'horizontal'

fig.show()

In [48]:
# 트리맵
fig = go.Figure()

fig.add_trace(go.Treemap(
    # 트레이스의 labels
    labels = name1 +
             name2 +
             name3,
    # 트레이스의 parents
    parents = [''] +
              ['전체'] * len(name2) +
              중계열_sum['대계열'].tolist(),
    # 트레이스의 values
    values = [전체_sum] +
             계열_sum +
             중계열_sum['졸업자수'].values.tolist(),
    textinfo = 'label+value+percent parent+percent entry'))

fig.show()

### Q3: (대)계열별 취업률은 어떻게 다를까? - 분포의 비교

In [49]:
fig = go.Figure()

fig.add_trace(go.Box(
    x = df_취업률['대계열'],
    y = df_취업률['취업률_계'],
    boxmean = 'sd'))

fig.update_layout(title = dict(text = '대학 계열별 취업률 분포', x = 0.5))

fig.show()

In [50]:
# 대학종류별로 더 세분화
fig = go.Figure()

fig.add_trace(go.Box(
    x = df_취업률.loc[df_취업률['과정구분'] == '전문대학과정', '대계열'],
    y = df_취업률['취업률_계'],
    name = '전문대학과정'
))

fig.add_trace(go.Box(
    x = df_취업률.loc[df_취업률['과정구분'] == '대학과정', '대계열'],
    y = df_취업률['취업률_계'],
    name = '대학과정'
))

fig.add_trace(go.Box(
    x = df_취업률.loc[df_취업률['과정구분'] == '대학원과정', '대계열'],
    y = df_취업률['취업률_계'],
    name = '대학원과정'
))

## boxmode를 group으로 설정
fig.update_layout(boxmode = 'group',
                  title = dict(text = '학위과정별 취업률 분포', x = 0.5, xanchor = 'center'))


In [51]:
# 바이올린 플랏
fig = go.Figure()

fig.add_trace(go.Violin(
    x = df_취업률['대계열'], y = df_취업률['취업률_계'],
    box = dict(visible = True),        # 박스 플랏 표시
    meanline = dict(visible = True)))  # 평균 선 표시


fig.update_layout(title = dict(text = '대학 계열별 취업률 분포', x = 0.5))

fig.show()

In [52]:
# 전문대와 대학으로 구분
fig = go.Figure()

## 전문대학과정 violin 트레이스 추가
fig.add_trace(go.Violin(
    x = df_취업률.loc[df_취업률['과정구분'] == '전문대학과정', '대계열'],
    y = df_취업률['취업률_계'],
    name = '전문대학',
    side = 'negative', box = dict(visible = True, width = 0.5),
    meanline = dict(visible = True, width = 1))
             )

fig.add_trace(go.Violin(
    x = df_취업률.loc[df_취업률['과정구분'] == '대학과정', '대계열'],
    y = df_취업률['취업률_계'],
    name = '대학',
    side ='positive', box = dict(visible = True, width = 0.5),
    meanline = dict(visible = True, width = 1))
             )

fig.update_layout(violinmode = "overlay",
        title = dict(text = '대학 계열별 취업률 분포', x = 0.5))

# Example 3 - 주가 데이터


### Q1: 지난 100일간의 삼성전자 주가는? - 캔들스틱 차트

In [54]:
!pip3 install -U finance-datareader

import FinanceDataReader as fdr
from datetime import datetime, timedelta

start_day = datetime(2022, 10, 7)
end_day = start_day + timedelta(days = 100)

samsung_stock = fdr.DataReader('005930', start_day, end_day)
samsung_stock.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 3.8 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [finance-datareader]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


,Open,High,Low,Close,Volume,Change
Date,,,,,,
2022-10-07,55900,56900,55200,56200,16886813,-0.001776
2022-10-11,54400,55700,54000,55400,21437877,-0.014235
2022-10-12,55700,57000,55200,55800,18408910,0.007220
2022-10-13,55400,56100,55200,55200,13784602,-0.010753
2022-10-14,56200,56500,55800,56300,12924326,0.019928


In [55]:
fig = go.Figure()

fig.add_trace(go.Candlestick(
    x = samsung_stock.index,
    open = samsung_stock['Open'],
    close = samsung_stock['Close'],
    high = samsung_stock['High'],
    low = samsung_stock['Low'],
    increasing = dict(line = dict(color = 'red')),   # 상승 색
    decreasing = dict(line = dict(color = 'blue')))) # 하락 색

fig.update_layout(title = dict(text = "삼성전자 Candlestick Chart", x = 0.5),
                  xaxis = dict(rangeslider = dict(visible = False)))

fig.show()

### Q2. 거래량은? (Subplot)

In [56]:
from plotly.subplots import make_subplots
fig = make_subplots(rows = 2, cols = 1, row_heights=[0.7, 0.3], shared_xaxes = True)

fig.add_trace(go.Candlestick(
    x = samsung_stock.index,
    open = samsung_stock['Open'], close = samsung_stock['Close'],
    high = samsung_stock['High'], low = samsung_stock['Low'],
    increasing = dict(line = dict(color = 'red')),
    decreasing = dict(line = dict(color = 'blue'))),
    row = 1, col = 1)

## 거래량 막대그래프인 bar 트레이스 추가
fig.add_trace(go.Bar(
    x = samsung_stock.index,
    y = samsung_stock['Volume'],
    marker = dict(color = 'gray')),
    row = 2, col =1)

## 서브플롯들의 Y축 제목 설정
fig.update_yaxes(title_text = "주가", row = 1, col = 1)
fig.update_yaxes(title_text = "거래량", row = 2, col = 1)

fig.update_layout(
    title = dict(text = "삼성전자 Candlestick Chart", x = 0.5),
    xaxis = dict(rangeslider = dict(visible = False)),
    showlegend = False)

fig.show()

In [57]:
# 이동평균선 추가
## 5일, 20일, 40일 이동평균 산출
samsung_stock['M5'] = samsung_stock['Close'].rolling(5).mean()
samsung_stock['M20'] = samsung_stock['Close'].rolling(20).mean()
samsung_stock['M40'] = samsung_stock['Close'].rolling(40).mean()
samsung_stock

## 서브플롯 설정
fig = make_subplots(rows = 2, cols = 1, row_heights=[0.7, 0.3])

## candlestick 트레이스 생성
fig.add_trace(
    go.Candlestick(
        x = samsung_stock.index,
        open = samsung_stock['Open'],
        close = samsung_stock['Close'],
        high = samsung_stock['High'],
        low = samsung_stock['Low'],
        increasing = dict(line = dict(color = 'red')),
        decreasing = dict(line = dict(color = 'blue')),
        showlegend = False),
    row = 1, col = 1)

## 5일 이동평균
fig.add_trace(
    go.Scatter(
        mode = 'lines',
        x = samsung_stock.index,
        y = samsung_stock['M5'],
        name = '5일 이동평균',
        line = dict(dash = 'solid')),
    row = 1, col = 1)

## 20일 이동평균
fig.add_trace(
    go.Scatter(
        mode = 'lines',
        x = samsung_stock.index,
        y = samsung_stock['M20'],
        name = '20일 이동평균',
        line = dict(dash = 'dash')),
    row = 1, col = 1)

## 40일 이동평균
fig.add_trace(
    go.Scatter(
        mode = 'lines',
        x = samsung_stock.index,
        y = samsung_stock['M40'],
        name = '40일 이동평균',
        line = dict(dash = 'dot')),
    row = 1, col = 1)

## 거래량 bar 트레이스 추가
fig.add_trace(
    go.Bar(
        x = samsung_stock.index,
        y = samsung_stock['Volume'],
        marker = dict(color = 'gray'),
        xaxis = 'x2', showlegend = False),
    row = 2, col =1)

fig.update_xaxes(
    rangeslider = dict(visible = False),
    rangebreaks = [
        dict(bounds = ["sat", "mon"]), # 주말제거
        dict(values = ["2022-09-09", "2022-09-12", "2022-10-03", "2022-10-10", "2022-12-30"])],
    row = 2, col = 1)

fig.update_yaxes(title_text = "거래량", row = 2, col = 1)

fig.update_layout(xaxis = dict(rangeslider = dict(visible = False),
    rangebreaks = [
        dict(bounds = ["sat", "mon"]),
        dict(values = ["2022-09-09", "2022-09-12", "2022-10-03", "2022-10-10", "2022-12-30"])]),
    yaxis = dict(title_text = "주가"),
    title = dict(text = "삼성전자 Candlestick Chart", x = 0.5),
    showlegend = False)

fig.show()